Setup & Mount

In [2]:
# ============================================================
# Coping in Crisis: Computational Analysis of Coping Styles
# in Digital Discourse During the 2023 Türkiye Earthquake
# CSSM 530 — Spring 2026 | Şevval Çakıcı | Koç University
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Dependencies

In [ ]:
!pip install transformers datasets scikit-learn torch statsmodels seaborn openai -q

import os, json, ast, random, time, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, hamming_loss, classification_report, cohen_kappa_score
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
from statsmodels.stats.contingency_tables import mcnemar
from openai import OpenAI

warnings.filterwarnings("ignore")
random.seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Data Loading & Phase Assignment

In [ ]:
# ── Configuration ──────────────────────────────────────────
DATA_ZIP  = "/content/drive/MyDrive/Copy of deprem_tweets_260401.zip"
DATA_DIR  = "/content/data"
DATA_FILE = f"{DATA_DIR}/deprem_tweets_260401.jsonl"
DRIVE_DIR = "/content/drive/MyDrive"

PHASE_BOUNDS = [
    (pd.Timestamp("2023-02-09"), 1),
    (pd.Timestamp("2023-02-13"), 2),
    (pd.Timestamp("2023-02-21"), 3),
]

def get_phase(date_str):
    d = pd.Timestamp(date_str)
    if d < pd.Timestamp("2023-02-09"):   return 1
    elif d < pd.Timestamp("2023-02-13"): return 2
    elif d < pd.Timestamp("2023-02-21"): return 3
    else:                                return 4

# Extract corpus
import zipfile
os.makedirs(DATA_DIR, exist_ok=True)
print("Extracting corpus...")
with zipfile.ZipFile(DATA_ZIP, 'r') as z:
    z.extractall(DATA_DIR)
print(f"Done. Files: {os.listdir(DATA_DIR)}")

Stratified Sampling (500 Tweets, 4 Phases)

In [ ]:
TARGET_PER_PHASE = 200
FINAL_PER_PHASE  = 125
buckets = {1: [], 2: [], 3: [], 4: []}

print("Scanning corpus for stratified sample...")
with open(DATA_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        if obj.get("lang") != "tr":               continue
        if obj.get("text", "").startswith("RT "): continue
        if len(obj.get("text", "")) < 30:         continue

        phase  = get_phase(obj["date"])
        bucket = buckets[phase]
        if len(bucket) < TARGET_PER_PHASE:
            bucket.append(obj)
        else:
            j = random.randint(0, i)
            if j < TARGET_PER_PHASE:
                bucket[j] = obj

        if i % 500_000 == 0:
            counts = {k: len(v) for k, v in buckets.items()}
            print(f"  {i:,} rows scanned | Phase counts: {counts}")

print(f"Scan complete. Per-phase counts: { {k: len(v) for k, v in buckets.items()} }")

# Finalize sample
rows = []
for phase, bucket in buckets.items():
    for obj in random.sample(bucket, FINAL_PER_PHASE):
        rows.append({
            "tweet_id":         obj["_id"],
            "date":             obj["date"],
            "text":             obj["text"],
            "phase":            phase,
            "emotions":         obj.get("emotions", []),
            "like_count":       obj.get("like_count", 0),
            "retweet_count":    obj.get("retweet_count", 0),
            "impression_count": obj.get("impression_count", 0),
            "reply_count":      obj.get("reply_count", 0),
            "gender":           obj.get("gender", ""),
            "org":              obj.get("org", ""),
            "matched_keywords": obj.get("matched_keywords", []),
        })

df_sample = (pd.DataFrame(rows)
               .sample(frac=1, random_state=42)
               .reset_index(drop=True))
df_sample["tweet_index"] = df_sample.index + 1

print(f"\nFinal sample: {len(df_sample)} tweets")
print(df_sample["phase"].value_counts().sort_index())
df_sample.to_csv(f"{DRIVE_DIR}/cssm530_sample_500.csv",
                 index=False, encoding="utf-8-sig")

GPT-4o Annotation

In [ ]:
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

ANNOTATION_PROMPT = """You are annotating tweets for a social science study.
The tweet was posted after the February 6, 2023 earthquake in Türkiye.

Following Lazarus & Folkman (1984) coping theory, assign 1 (present) or 0 (absent)
to each category. A tweet may receive multiple labels (multi-label scheme).

CATEGORIES:
1. problem_focused: Information sharing, aid coordination, resource referral, practical solutions.
   Examples: sharing debris coordinates, AFAD numbers, donation links.

2. emotion_focused: Fear, grief, gratitude, helplessness, hope, or emotional support.
   Examples: condolences, prayers, expressions of sorrow or relief.

3. meaning_making: Moral, religious, ideological, or societal framing. Blame attribution.
   Examples: political criticism, religious fatalism, calls for accountability.

4. avoidance: Deflection, irony, humor, distancing, or topic shifts.
   Examples: sarcasm, unrelated content, dismissiveness.

Return ONLY a JSON object, no other text:
{"problem_focused": 0 or 1, "emotion_focused": 0 or 1, "meaning_making": 0 or 1,
 "avoidance": 0 or 1, "confidence": "high/medium/low", "reasoning": "one sentence"}"""

def annotate_tweet(text):
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": ANNOTATION_PROMPT},
                {"role": "user",   "content": f"Tweet: {text}"}
            ],
            temperature=0,
            max_tokens=200
        )
        content = (response.choices[0].message.content
                   .strip().replace("```json","").replace("```","").strip())
        return json.loads(content)
    except Exception as e:
        return {"problem_focused": -1, "emotion_focused": -1,
                "meaning_making": -1, "avoidance": -1,
                "confidence": "error", "reasoning": str(e)}

df_sample = pd.read_csv(f"{DRIVE_DIR}/cssm530_sample_500.csv")
results   = []

print("Starting GPT-4o annotation (500 tweets)...")
for i, row in df_sample.iterrows():
    result = annotate_tweet(row["text"])
    result["tweet_index"] = row["tweet_index"]
    results.append(result)
    time.sleep(0.3)
    if (i + 1) % 50 == 0:
        pd.DataFrame(results).to_csv(
            f"{DRIVE_DIR}/gpt4o_annotations_temp.csv", index=False)
        print(f"  {i+1}/500 | Errors: {sum(1 for r in results if r['problem_focused']==-1)}")

df_gpt = pd.DataFrame(results).rename(columns={
    "problem_focused": "gpt_problem",  "emotion_focused": "gpt_emotion",
    "meaning_making":  "gpt_meaning",  "avoidance":       "gpt_avoidance",
    "confidence":      "gpt_confidence","reasoning":       "gpt_reasoning"
})
df_annotated = df_sample.merge(df_gpt, on="tweet_index")
df_annotated.to_csv(f"{DRIVE_DIR}/cssm530_gpt_annotated.csv",
                    index=False, encoding="utf-8-sig")

print(f"\nAnnotation complete. Label distribution:")
for col in ["gpt_problem","gpt_emotion","gpt_meaning","gpt_avoidance"]:
    print(f"  {col}: {df_annotated[col].sum()} ({df_annotated[col].mean()*100:.1f}%)")

Inter-Annotator Agreement (Cohen's Kappa)

In [ ]:
EXCEL_PATH = f"{DRIVE_DIR}/cssm530_manual_annotation_50.xlsx"

df_gpt_iaa = pd.read_excel(EXCEL_PATH, sheet_name="Sheet1")
df_h1       = pd.read_excel(EXCEL_PATH, sheet_name="Human1")
df_h2       = pd.read_excel(EXCEL_PATH, sheet_name="Human2")
df_h3       = pd.read_excel(EXCEL_PATH, sheet_name="Human3")

df_gpt_iaa = df_gpt_iaa.rename(columns={
    "gpt_problem":"problem","gpt_emotion":"emotion",
    "gpt_meaning":"meaning","gpt_avoidance":"avoidance"
})

labels     = ["problem", "emotion", "meaning", "avoidance"]
annotators = {"H1": df_h1, "H2": df_h2, "H3": df_h3, "GPT": df_gpt_iaa}

def align(df_a, df_b, cols):
    return df_a[["tweet_index"]+cols].merge(
        df_b[["tweet_index"]+cols], on="tweet_index",
        suffixes=("_a","_b")).dropna()

print(f"{'Pair':<12}", *[f"{l:>10}" for l in labels], f"{'Mean':>10}")
print("=" * 62)

records = []
for a, b in combinations(annotators.keys(), 2):
    merged = align(annotators[a], annotators[b], labels)
    row    = {"pair": f"{a} vs {b}"}
    kappas = []
    for label in labels:
        try:
            k = cohen_kappa_score(
                merged[f"{label}_a"].astype(int),
                merged[f"{label}_b"].astype(int))
        except Exception:
            k = np.nan
        row[label] = k
        kappas.append(k)
    row["mean"] = np.nanmean(kappas)
    records.append(row)
    print(f"{row['pair']:<12}", *[f"{row[l]:>10.3f}" for l in labels],
          f"{row['mean']:>10.3f}")

df_kappa = pd.DataFrame(records)
print("=" * 62)
print("Overall mean κ per label:")
for label in labels:
    print(f"  {label}: κ = {df_kappa[label].mean():.3f}")
print(f"  Overall: κ = {df_kappa['mean'].mean():.3f}")

df_kappa.to_csv(f"{DRIVE_DIR}/cssm530_kappa_results.csv", index=False)
print("\nNote: 'avoidance' excluded from downstream analysis (κ = 0.021)")

Gold Standard Label Construction

In [ ]:
df_h1  = pd.read_excel(EXCEL_PATH, sheet_name="Human1")
df_h2  = pd.read_excel(EXCEL_PATH, sheet_name="Human2")
df_h3  = pd.read_excel(EXCEL_PATH, sheet_name="Human3")
df_gpt = pd.read_csv(f"{DRIVE_DIR}/cssm530_gpt_annotated.csv")

df_gpt = df_gpt.rename(columns={
    "gpt_problem":"problem","gpt_emotion":"emotion","gpt_meaning":"meaning"
})[["tweet_index","problem","emotion","meaning"]]

labels = ["problem","emotion","meaning"]

def get_labels(df, suffix):
    return df[["tweet_index"]+labels].rename(
        columns={l: f"{l}_{suffix}" for l in labels})

merged = get_labels(df_h1, "h1")
for df, sfx in [(df_h2,"h2"),(df_h3,"h3"),(df_gpt,"gpt")]:
    merged = merged.merge(get_labels(df, sfx), on="tweet_index", how="inner")

# Majority vote (2/3 human annotators)
for label in labels:
    votes = (merged[f"{label}_h1"].astype(int) +
             merged[f"{label}_h2"].astype(int) +
             merged[f"{label}_h3"].astype(int))
    merged[f"gold_{label}"] = (votes >= 2).astype(int)

df_full = (pd.read_csv(f"{DRIVE_DIR}/cssm530_gpt_annotated.csv")
             .rename(columns={"gpt_problem":"gold_problem",
                               "gpt_emotion":"gold_emotion",
                               "gpt_meaning":"gold_meaning"}))
gold_50 = merged[["tweet_index","gold_problem","gold_emotion","gold_meaning"]]

df_full = df_full.set_index("tweet_index")
df_full.update(gold_50.set_index("tweet_index"))
df_full = df_full.reset_index()

df_full.to_csv(f"{DRIVE_DIR}/cssm530_gold_standard.csv",
               index=False, encoding="utf-8-sig")
print(f"Gold standard dataset: {len(df_full)} tweets")
for col in ["gold_problem","gold_emotion","gold_meaning"]:
    print(f"  {col}: {df_full[col].sum()} ({df_full[col].mean()*100:.1f}%)")

BERTurk Fine-Tuning

In [ ]:
MODEL_NAME = "dbmdz/bert-base-turkish-cased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
LABEL_COLS = ["gold_problem","gold_emotion","gold_meaning"]

df = pd.read_csv(f"{DRIVE_DIR}/cssm530_gold_standard.csv")
df = df.dropna(subset=["text"]+LABEL_COLS)
df[LABEL_COLS] = df[LABEL_COLS].astype(int)

train_df, temp_df = train_test_split(df, test_size=0.30,
                                     random_state=42, stratify=df["phase"])
dev_df,   test_df = train_test_split(temp_df, test_size=0.50,
                                     random_state=42, stratify=temp_df["phase"])
print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")

def tokenize(examples):
    tokens = tokenizer(examples["text"], padding="max_length",
                       truncation=True, max_length=128)
    tokens["labels"] = [
        [float(examples[col][i]) for col in LABEL_COLS]
        for i in range(len(examples["text"]))
    ]
    return tokens

def to_hf_dataset(df):
    ds = Dataset.from_pandas(df[["text"]+LABEL_COLS].reset_index(drop=True))
    ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
    ds.set_format("torch")
    return ds

train_ds = to_hf_dataset(train_df)
dev_ds   = to_hf_dataset(dev_df)
test_ds  = to_hf_dataset(test_df)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, problem_type="multi_label_classification")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.sigmoid(torch.tensor(logits)).numpy()
    preds  = (probs >= 0.5).astype(int)
    labels = labels.astype(int)
    per_label = f1_score(labels, preds, average=None, zero_division=0)
    return {
        "macro_f1":        round(f1_score(labels, preds, average="macro",  zero_division=0), 4),
        "hamming_loss":    round(hamming_loss(labels, preds), 4),
        "subset_accuracy": round(float(np.mean(np.all(preds==labels, axis=1))), 4),
        "f1_problem":      round(per_label[0], 4),
        "f1_emotion":      round(per_label[1], 4),
        "f1_meaning":      round(per_label[2], 4),
    }

training_args = TrainingArguments(
    output_dir="/content/bertturk_coping",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    seed=42,
    report_to="none"
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_ds, eval_dataset=dev_ds,
                  compute_metrics=compute_metrics)

print("Training BERTurk...")
trainer.train()
print("Training complete.")

Test Set Evaluation

In [ ]:
LABEL_NAMES = ["problem_focused","emotion_focused","meaning_making"]

predictions = trainer.predict(test_ds)
probs_bert  = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
preds_bert  = (probs_bert >= 0.5).astype(int)
true_labels = predictions.label_ids.astype(int)

macro_f1   = f1_score(true_labels, preds_bert, average="macro",  zero_division=0)
hamming    = hamming_loss(true_labels, preds_bert)
subset_acc = float(np.mean(np.all(preds_bert==true_labels, axis=1)))
per_label  = f1_score(true_labels, preds_bert, average=None, zero_division=0)

print("TEST SET RESULTS — BERTurk")
print(f"  Macro F1:        {macro_f1:.4f}")
print(f"  Hamming Loss:    {hamming:.4f}")
print(f"  Subset Accuracy: {subset_acc:.4f}")
for name, f1 in zip(LABEL_NAMES, per_label):
    print(f"  F1 {name}: {f1:.4f}")
print()
print(classification_report(true_labels, preds_bert,
                             target_names=LABEL_NAMES, zero_division=0))

pd.DataFrame({"metric":["macro_f1","hamming_loss","subset_accuracy",
                         "f1_problem","f1_emotion","f1_meaning"],
              "value": [macro_f1, hamming, subset_acc,
                        per_label[0], per_label[1], per_label[2]]
}).to_csv(f"{DRIVE_DIR}/cssm530_test_results.csv", index=False)

Zero-Shot Baseline (mDeBERTa)

In [ ]:
from transformers import pipeline as hf_pipeline

zs_pipeline = hf_pipeline("zero-shot-classification",
                           model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
                           device=0)

HYPOTHESES = [
    "This tweet contains information sharing, aid coordination, or practical help.",
    "This tweet expresses grief, fear, gratitude, or emotional support.",
    "This tweet frames the crisis through moral, religious, or political meaning.",
]

test_texts  = test_df["text"].tolist()
test_labels_np = test_df[["gold_problem","gold_emotion","gold_meaning"]].values.astype(int)

preds_zs = []
for i, text in enumerate(test_texts):
    try:
        result = zs_pipeline(text[:512], HYPOTHESES, multi_label=True)
        scores = dict(zip(result["labels"], result["scores"]))
        preds_zs.append([int(scores[h] >= 0.5) for h in HYPOTHESES])
    except Exception:
        preds_zs.append([0, 0, 0])
    if (i+1) % 25 == 0:
        print(f"  {i+1}/{len(test_texts)} completed")

preds_zs   = np.array(preds_zs)
zs_macro   = f1_score(test_labels_np, preds_zs, average="macro",  zero_division=0)
zs_per     = f1_score(test_labels_np, preds_zs, average=None,     zero_division=0)
zs_hamming = hamming_loss(test_labels_np, preds_zs)

print("\nMODEL COMPARISON")
print(f"{'Metric':<20} {'BERTurk':>10} {'Zero-Shot':>12} {'Δ':>8}")
print("-" * 52)
for m, b, z in zip(["Macro F1","F1 Problem","F1 Emotion","F1 Meaning"],
                    [macro_f1, per_label[0], per_label[1], per_label[2]],
                    [zs_macro,  zs_per[0],   zs_per[1],   zs_per[2]]):
    print(f"{m:<20} {b:>10.3f} {z:>12.3f} {b-z:>+8.3f}")

pd.DataFrame({"metric":["macro_f1","f1_problem","f1_emotion","f1_meaning"],
              "bertturk":[macro_f1,per_label[0],per_label[1],per_label[2]],
              "zeroshot": [zs_macro, zs_per[0],  zs_per[1],  zs_per[2]]
}).to_csv(f"{DRIVE_DIR}/cssm530_model_comparison.csv", index=False)

Full Corpus Inference

In [ ]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PRED_COLS   = ["pred_problem","pred_emotion","pred_meaning"]
BATCH_SIZE  = 64
results_inf = []
batch_texts, batch_meta = [], []
total = 0

print("Running inference on full corpus...")
with open(DATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("lang") != "tr":               continue
        if obj.get("text","").startswith("RT "):  continue
        if len(obj.get("text","")) < 20:          continue

        batch_texts.append(obj["text"])
        batch_meta.append({
            "tweet_id":         obj["_id"],
            "date":             obj["date"],
            "phase":            get_phase(obj["date"]),
            "like_count":       obj.get("like_count", 0),
            "retweet_count":    obj.get("retweet_count", 0),
            "impression_count": obj.get("impression_count", 0),
            "emotions":         str(obj.get("emotions", [])),
        })

        if len(batch_texts) == BATCH_SIZE:
            enc   = tokenizer(batch_texts, padding=True, truncation=True,
                              max_length=128, return_tensors="pt").to(device)
            with torch.no_grad():
                logits = model(**enc).logits
            preds = (torch.sigmoid(logits).cpu().numpy() >= 0.5).astype(int)
            for meta, pred in zip(batch_meta, preds):
                meta.update(dict(zip(PRED_COLS, pred)))
                results_inf.append(meta)
            batch_texts, batch_meta = [], []
            total += BATCH_SIZE
            if total % 50_000 == 0:
                print(f"  {total:,} tweets processed")

if batch_texts:
    enc   = tokenizer(batch_texts, padding=True, truncation=True,
                      max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    preds = (torch.sigmoid(logits).cpu().numpy() >= 0.5).astype(int)
    for meta, pred in zip(batch_meta, preds):
        meta.update(dict(zip(PRED_COLS, pred)))
        results_inf.append(meta)

df_corpus = pd.DataFrame(results_inf)
df_corpus.to_csv(f"{DRIVE_DIR}/cssm530_corpus_predictions.csv", index=False)
print(f"\nInference complete: {len(df_corpus):,} tweets")
for col in PRED_COLS:
    print(f"  {col}: {df_corpus[col].sum():,} ({df_corpus[col].mean()*100:.1f}%)")

Visualizations

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
})

C = {"problem":"#5B9BD5","emotion":"#E07B8A","meaning":"#F5A962",
     "gray":"#888888","green":"#8DBB8D"}

df_corpus = pd.read_csv(f"{DRIVE_DIR}/cssm530_corpus_predictions.csv")
df_corpus["date"] = pd.to_datetime(df_corpus["date"])
df_corpus["day"]  = df_corpus["date"].dt.date

PRED_COLS  = ["pred_problem","pred_emotion","pred_meaning"]
PRED_NAMES = ["Problem-Focused","Emotion-Focused","Meaning-Making"]
COLORS     = [C["problem"],C["emotion"],C["meaning"]]

# Figure 1 — Temporal distribution
daily = df_corpus.groupby("day")[PRED_COLS].mean().reset_index()
daily["day"] = pd.to_datetime(daily["day"])

fig, ax = plt.subplots(figsize=(12, 4.5), facecolor="white")
ax.set_facecolor("white")
for col, name, color in zip(PRED_COLS, PRED_NAMES, COLORS):
    ax.plot(daily["day"], daily[col], color=color,
            linewidth=2.2, alpha=0.9, label=name)

phase_bands = [
    ("2023-02-06","2023-02-09",C["problem"],"Phase 1\nUrgency"),
    ("2023-02-09","2023-02-13",C["emotion"],"Phase 2\nBlame"),
    ("2023-02-13","2023-02-21",C["meaning"],"Phase 3\nGrief"),
    ("2023-02-21","2023-03-07",C["green"],  "Phase 4\nMeaning"),
]
for start, end, color, label in phase_bands:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.06, color=color)
    mid = pd.Timestamp(start) + (pd.Timestamp(end)-pd.Timestamp(start))/2
    ax.text(mid, 0.67, label, ha="center", va="top",
            fontsize=8.5, color=C["gray"], style="italic")

ax.set_xlabel("Date", fontsize=11, labelpad=8)
ax.set_ylabel("Mean Coping Rate", fontsize=11, labelpad=8)
ax.set_title("Daily Coping Style Distribution Following the\nFebruary 6, 2023 Earthquake in Türkiye",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=10, framealpha=0.8, loc="upper right")
ax.set_ylim(0.05, 0.72)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/fig1_temporal_coping.png")
plt.show()

# Figure 2 — Phase bar chart
phase_means = df_corpus.groupby("phase")[PRED_COLS].mean()
phase_n     = df_corpus["phase"].value_counts().sort_index()
x, width    = np.arange(4), 0.26
XLABELS     = ["Phase 1\nUrgency","Phase 2\nBlame",
               "Phase 3\nGrief","Phase 4\nMeaning-Making"]

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor="white")
ax.set_facecolor("white")
for i, (col, name, color) in enumerate(zip(PRED_COLS, PRED_NAMES, COLORS)):
    bars = ax.bar(x+i*width, phase_means[col], width, label=name,
                  color=color, alpha=0.88, edgecolor="white")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.008,
                f"{h:.2f}", ha="center", va="bottom", fontsize=8, color="#555555")
for i, (_, n) in enumerate(phase_n.items()):
    ax.text(i+width, -0.045, f"n={n/1000:.0f}K",
            ha="center", fontsize=8, color=C["gray"])
ax.set_xticks(x+width)
ax.set_xticklabels(XLABELS, fontsize=10.5)
ax.set_ylabel("Mean Coping Rate", fontsize=11, labelpad=8)
ax.set_title("Coping Style Distribution Across Crisis Phases",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=10, framealpha=0.8)
ax.set_ylim(0, 0.72)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/fig2_phase_coping.png")
plt.show()

# Figure 3 — Engagement correlations
fig, axes = plt.subplots(1, 3, figsize=(13, 4), facecolor="white")
for ax, ecol, elabel in zip(axes,
    ["like_count","retweet_count","impression_count"],
    ["Likes","Retweets","Impressions"]):
    ax.set_facecolor("white")
    corrs, pvals = zip(*[stats.spearmanr(df_corpus[c], df_corpus[ecol])
                         for c in PRED_COLS])
    bars = ax.bar(PRED_NAMES, corrs, color=COLORS,
                  alpha=0.85, edgecolor="white", width=0.55)
    ax.axhline(0, color="#333333", linewidth=0.9)
    ax.set_title(elabel, fontsize=11, fontweight="bold", pad=8)
    if ax is axes[0]: ax.set_ylabel("Spearman r", fontsize=10)
    ax.set_ylim(-0.20, 0.20)
    ax.tick_params(axis="x", labelsize=8.5, rotation=15)
    for bar, r, p in zip(bars, corrs, pvals):
        if p < 0.05:
            ax.text(bar.get_x()+bar.get_width()/2,
                    r+0.008 if r>=0 else r-0.016,
                    "✱", ha="center", fontsize=13, color="#333333")
fig.suptitle("Coping Styles and Engagement Metrics (Spearman r, ✱ p < .05)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/fig3_engagement.png")
plt.show()

# Figure 4 — Emotion heatmap
def parse_emotions(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else (x or [])
    except:
        return []

df_corpus["emotions_list"] = df_corpus["emotions"].apply(parse_emotions)
EMO_MAP = {"ofke":"Anger","korku":"Fear","uzuntu":"Sadness",
           "saskinlik":"Surprise","onaylamama":"Disapproval"}
for tr, en in EMO_MAP.items():
    df_corpus[f"emo_{en}"] = df_corpus["emotions_list"].apply(
        lambda x, t=tr: 1 if t in x else 0)

emo_labels = list(EMO_MAP.values())
hmap = pd.DataFrame(index=emo_labels, columns=PRED_NAMES, dtype=float)
amap = pd.DataFrame(index=emo_labels, columns=PRED_NAMES, dtype=str)
for en in emo_labels:
    for ccol, cname in zip(PRED_COLS, PRED_NAMES):
        r, p = stats.spearmanr(df_corpus[f"emo_{en}"], df_corpus[ccol])
        hmap.loc[en, cname] = round(r, 3)
        amap.loc[en, cname] = f"{r:.3f}{'✱' if p<0.05 else ''}"

fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
sns.heatmap(hmap.astype(float), annot=amap, fmt="",
            cmap="RdBu_r", center=0, vmin=-0.35, vmax=0.35,
            linewidths=0.6, linecolor="#eeeeee", annot_kws={"size":10},
            cbar_kws={"shrink":0.8,"label":"Spearman r"}, ax=ax)
ax.set_title("Politus Emotion Scores × Coping Styles (✱ p < .05)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Coping Style", fontsize=10, labelpad=8)
ax.set_ylabel("Emotion Category", fontsize=10, labelpad=8)
plt.xticks(rotation=15, fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/fig4_emotion_heatmap.png")
plt.show()

print("All figures saved.")